# Exporting Embedding Tiles as GeoTIFFs

This tutorial demonstrates how to export Tessera embedding tiles as individual GeoTIFF files using the new export APIs. Each tile is exported with proper georeferencing using the landmask CRS and transform.

## Key Features
- Export individual tiles as GeoTIFFs
- Use landmask CRS/transform for accurate georeferencing
- Support for different ROI input types (geometry, bounds, file)
- Band selection options
- Full 128-band or selective band export

In [1]:
import os
import tempfile
from pathlib import Path
import rasterio
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
import numpy as np

from geotessera import GeoTessera

## Initialize GeoTessera

In [2]:
# Initialize GeoTessera
gt = GeoTessera()
print("✅ GeoTessera initialized")

✅ GeoTessera initialized


In [3]:
import os
os.environ["TESSERA_DATA_DIR"] = "/scratch/jk871/.cache/tessera"  # or whatever path you prefer

## Define Region of Interest

We'll use the Cambridge UK bounds.

In [16]:
# India region bounds (west, south, east, north)
roi_bounds = (0.058365,52.148568,0.196918,52.239364)
print(f"ROI bounds: {roi_bounds}")

# Create a Shapely geometry for the region
roi_geometry = box(*roi_bounds)
print(f"Region area: {roi_geometry.area:.6f} square degrees")

ROI bounds: (0.058365, 52.148568, 0.196918, 52.239364)
Region area: 0.012580 square degrees


## Method 1: Export using Shapely Geometry

In [17]:
# Create a temporary directory for exports
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Using temporary directory: {temp_dir}")
    
    # Export tiles using geometry
    print("\n🔄 Exporting tiles using geometry...")
    tiff_paths = gt.export_embedding_tiles_for_region(
        geometry=roi_geometry,
        output_dir=os.path.join(temp_dir, "geometry_export"),
        bands=[0, 1, 2],  # RGB visualization
        year=2024
    )
    
    print(f"✅ Exported {len(tiff_paths)} tiles")
    
    # Display information about each exported tile
    print("\n📊 Tile Information:")
    for i, tiff_path in enumerate(tiff_paths):
        with rasterio.open(tiff_path) as src:
            print(f"  Tile {i+1}: {os.path.basename(tiff_path)}")
            print(f"    CRS: {src.crs}")
            print(f"    Bounds: {src.bounds}")
            print(f"    Shape: {src.shape}")
            print(f"    Transform: {src.transform}")
            print(f"    File size: {os.path.getsize(tiff_path) / 1024 / 1024:.1f} MB")
            print()

Using temporary directory: /tmp/tmpas0yp98h

🔄 Exporting tiles using geometry...
Exporting 3 selected bands
Loading 1 registry blocks for region bounds: (0.0584, 52.1486, 0.1969, 52.2394)
Using 1/1 already loaded registry blocks
Found 4 embedding tiles to export for year 2024
Exported tile at (52.15, 0.05) to /tmp/tmpas0yp98h/geometry_export/tile_52.15_0.05.tiff
Exported tile at (52.15, 0.15) to /tmp/tmpas0yp98h/geometry_export/tile_52.15_0.15.tiff
Exported tile at (52.25, 0.05) to /tmp/tmpas0yp98h/geometry_export/tile_52.25_0.05.tiff
Exported tile at (52.25, 0.15) to /tmp/tmpas0yp98h/geometry_export/tile_52.25_0.15.tiff
Successfully exported 4 tiles to /tmp/tmpas0yp98h/geometry_export
✅ Exported 4 tiles

📊 Tile Information:
  Tile 1: tile_52.15_0.05.tiff
    CRS: EPSG:32631
    Bounds: BoundingBox(left=294530.57271446165, bottom=5776125.730892532, right=301820.57271446165, top=5787525.730892532)
    Shape: (1140, 729)
    Transform: | 10.00, 0.00, 294530.57|
| 0.00,-10.00, 5787525.73|

## Method 2: Export using Bounds Tuple

In [18]:
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Using temporary directory: {temp_dir}")
    
    # Export tiles using bounds
    print("\n🔄 Exporting tiles using bounds...")
    tiff_paths = gt.export_embedding_tiles_for_region_bounds(
        bounds=roi_bounds,
        output_dir=os.path.join(temp_dir, "bounds_export"),
        bands=[10, 20, 30],  # Different band combination
        year=2024
    )
    
    print(f"✅ Exported {len(tiff_paths)} tiles")
    
    # Verify same number of tiles
    print(f"\n📊 Exported {len(tiff_paths)} tiles with bands [10, 20, 30]")

Using temporary directory: /tmp/tmpphxkpheg

🔄 Exporting tiles using bounds...
Exporting 3 selected bands
Loading 1 registry blocks for region bounds: (0.0584, 52.1486, 0.1969, 52.2394)
Using 1/1 already loaded registry blocks
Found 4 embedding tiles to export for year 2024


Exported tile at (52.15, 0.05) to /tmp/tmpphxkpheg/bounds_export/tile_52.15_0.05.tiff
Exported tile at (52.15, 0.15) to /tmp/tmpphxkpheg/bounds_export/tile_52.15_0.15.tiff
Exported tile at (52.25, 0.05) to /tmp/tmpphxkpheg/bounds_export/tile_52.25_0.05.tiff
Exported tile at (52.25, 0.15) to /tmp/tmpphxkpheg/bounds_export/tile_52.25_0.15.tiff
Successfully exported 4 tiles to /tmp/tmpphxkpheg/bounds_export
✅ Exported 4 tiles

📊 Exported 4 tiles with bands [10, 20, 30]


## Method 3: Export using Region File

Create a GeoJSON file and use the file-based export method.

In [19]:
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Using temporary directory: {temp_dir}")
    
    # Create a GeoJSON file for the region
    geojson_data = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {"name": "India Region"},
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [[
                        [roi_bounds[0], roi_bounds[1]],
                        [roi_bounds[2], roi_bounds[1]],
                        [roi_bounds[2], roi_bounds[3]],
                        [roi_bounds[0], roi_bounds[3]],
                        [roi_bounds[0], roi_bounds[1]]
                    ]]
                }
            }
        ]
    }
    
    # Save GeoJSON file
    geojson_path = os.path.join(temp_dir, "india_region.geojson")
    import json
    with open(geojson_path, 'w') as f:
        json.dump(geojson_data, f)
    
    print(f"Created GeoJSON file: {geojson_path}")
    
    # Export tiles using file
    print("\n🔄 Exporting tiles using region file...")
    tiff_paths = gt.export_embedding_tiles_for_region_file(
        region_path=geojson_path,
        output_dir=os.path.join(temp_dir, "file_export"),
        bands=None,  # Export all 128 bands
        year=2024
    )
    
    if tiff_paths:
        print(f"✅ Exported {len(tiff_paths)} tiles with all 128 bands")
        
        # Show file sizes (should be much larger with all bands)
        total_size = sum(os.path.getsize(p) for p in tiff_paths)
        print(f"Total size: {total_size / 1024 / 1024:.1f} MB")
    else:
        print("❌ Export failed")

Using temporary directory: /tmp/tmp_t81xzwi
Created GeoJSON file: /tmp/tmp_t81xzwi/india_region.geojson

🔄 Exporting tiles using region file...
Exporting all 128 bands
Loading 1 registry blocks for region bounds: (0.0584, 52.1486, 0.1969, 52.2394)
Using 1/1 already loaded registry blocks
Found 4 embedding tiles to export for year 2024
Exported tile at (52.15, 0.05) to /tmp/tmp_t81xzwi/file_export/tile_52.15_0.05.tiff
Exported tile at (52.15, 0.15) to /tmp/tmp_t81xzwi/file_export/tile_52.15_0.15.tiff
Exported tile at (52.25, 0.05) to /tmp/tmp_t81xzwi/file_export/tile_52.25_0.05.tiff
Exported tile at (52.25, 0.15) to /tmp/tmp_t81xzwi/file_export/tile_52.25_0.15.tiff
Successfully exported 4 tiles to /tmp/tmp_t81xzwi/file_export
✅ Exported 4 tiles with all 128 bands
Total size: 1441.4 MB


## Compare Different Band Selections

Let's export the same region with different band combinations to see the differences.

In [21]:
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Using temporary directory: {temp_dir}")
    
    # Test different band combinations
    band_combinations = [
        ([0, 1, 2], "RGB-like"),
        (None, "All 128 bands")
    ]
    
    results = []
    
    for bands, description in band_combinations:
        print(f"\n🔄 Exporting with {description}...")
        
        output_dir = os.path.join(temp_dir, f"bands_{description.replace(' ', '_').replace('-', '_')}")
        
        tiff_paths = gt.export_embedding_tiles_for_region(
            geometry=roi_geometry,
            output_dir=output_dir,
            bands=bands,
            year=2024
        )
        
        if tiff_paths:
            total_size = sum(os.path.getsize(p) for p in tiff_paths)
            avg_size = total_size / len(tiff_paths)
            
            results.append({
                'description': description,
                'bands': bands,
                'num_tiles': len(tiff_paths),
                'total_size_mb': total_size / 1024 / 1024,
                'avg_size_mb': avg_size / 1024 / 1024
            })
            
            print(f"  ✅ {len(tiff_paths)} tiles, {total_size / 1024 / 1024:.1f} MB total")
        else:
            print(f"  ❌ Export failed")
    
    # Display comparison table
    print("\n📊 Band Selection Comparison:")
    print("-" * 80)
    print(f"{'Description':<15} {'Bands':<15} {'Tiles':<6} {'Total (MB)':<12} {'Avg (MB)':<10}")
    print("-" * 80)
    for result in results:
        bands_str = str(result['bands']) if result['bands'] else 'All'
        print(f"{result['description']:<15} {bands_str:<15} {result['num_tiles']:<6} {result['total_size_mb']:<12.1f} {result['avg_size_mb']:<10.1f}")

Using temporary directory: /tmp/tmphrjys29q

🔄 Exporting with RGB-like...
Exporting 3 selected bands
Loading 1 registry blocks for region bounds: (0.0584, 52.1486, 0.1969, 52.2394)
Using 1/1 already loaded registry blocks
Found 4 embedding tiles to export for year 2024
Exported tile at (52.15, 0.05) to /tmp/tmphrjys29q/bands_RGB_like/tile_52.15_0.05.tiff
Exported tile at (52.15, 0.15) to /tmp/tmphrjys29q/bands_RGB_like/tile_52.15_0.15.tiff
Exported tile at (52.25, 0.05) to /tmp/tmphrjys29q/bands_RGB_like/tile_52.25_0.05.tiff
Exported tile at (52.25, 0.15) to /tmp/tmphrjys29q/bands_RGB_like/tile_52.25_0.15.tiff
Successfully exported 4 tiles to /tmp/tmphrjys29q/bands_RGB_like
  ✅ 4 tiles, 45.6 MB total

🔄 Exporting with All 128 bands...
Exporting all 128 bands
Loading 1 registry blocks for region bounds: (0.0584, 52.1486, 0.1969, 52.2394)
Using 1/1 already loaded registry blocks
Found 4 embedding tiles to export for year 2024
Exported tile at (52.15, 0.05) to /tmp/tmphrjys29q/bands_All_1

## Summary

This tutorial demonstrated:

1. **Three export methods**:
   - `export_embedding_tiles_for_region()` - using Shapely geometry
   - `export_embedding_tiles_for_region_bounds()` - using bounds tuple
   - `export_embedding_tiles_for_region_file()` - using region file

2. **Band selection options**:
   - Specific bands for visualization (e.g., [0,1,2] for RGB)
   - All 128 bands for full analysis

3. **Georeferencing**:
   - Each tile uses the landmask's native CRS and transform
   - Accurate geospatial positioning
   - No coordinate distortion

4. **File management**:
   - Individual GeoTIFF files per tile
   - Proper compression and metadata
   - Consistent naming convention

The exported GeoTIFFs can be used in GIS software like QGIS, ArcGIS, or for further analysis in Python with rasterio.